# 02 Image Processing Fundamentals and Feature Extraction

## 📚 Learning Objectives

By completing this notebook, you will:
- Train a baseline classifier with scikit-learn
- Evaluate using accuracy + confusion matrix
- Show how to encode categorical features

## 🔗 Prerequisites

- ✅ Python basics
- ✅ Jupyter Notebook basics

---

## Official Structure Reference

This notebook covers practical activities from **Course 08, Unit 2**:
- image processing fundamentals and feature extraction
- **Source:** `DETAILED_UNIT_DESCRIPTIONS.md`

**📌 Covers slide(s):** **10** — Image Processing Basics (pixels, CIFAR-10, augmentation). *Do this notebook after that slide.*

---


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Dataset:** Synthetic — small generated data (no download; for classifier/feature demo).

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [1]:
# Baseline classifier: synthetic data, OneHot for categorical, pipeline, evaluate
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# small synthetic dataset
rng = np.random.default_rng(123)
n = 600
x1 = rng.normal(size=n)
x2 = rng.normal(size=n)
color = rng.choice(['red','green','blue'], size=n)

y = ((x1 + 0.8*x2 + (color == 'red')*0.6 + rng.normal(scale=0.5, size=n)) > 0.2).astype(int)

df = pd.DataFrame({'x1': x1, 'x2': x2, 'color': color, 'y': y})
X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pre = ColumnTransformer([
 ('cat', OneHotEncoder(handle_unknown='ignore'), ['color']),
], remainder='passthrough')

clf = Pipeline([
 ('pre', pre),
 ('model', LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print('accuracy:', accuracy_score(y_test, y_pred))
print('confusion matrix:', confusion_matrix(y_test, y_pred))
print('\nreport:', classification_report(y_test, y_pred))


accuracy: 0.9266666666666666
confusion matrix: [[70  5]
 [ 6 69]]

report:               precision    recall  f1-score   support

           0       0.92      0.93      0.93        75
           1       0.93      0.92      0.93        75

    accuracy                           0.93       150
   macro avg       0.93      0.93      0.93       150
weighted avg       0.93      0.93      0.93       150



/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept


## 🌍 Real-World Worked Example — Handwritten Digit Recognition

**Industry context:** The US Postal Service has been using digit recognition since the 1980s.  
ATMs, banks, and postal systems worldwide classify millions of handwritten digits daily.

Below we train a neural network on **real** digit images (same family as MNIST, 8×8 pixels, sklearn built-in — no download needed).

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, matplotlib.pyplot as plt

# ── Real digit images (1797 samples, 8×8 pixels) ──────────────────────────
digits = load_digits()
X = StandardScaler().fit_transform(digits.data.astype(np.float32))
y = digits.target

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

X_tr_t = torch.tensor(X_tr)
y_tr_t = torch.tensor(y_tr, dtype=torch.long)
X_te_t = torch.tensor(X_te)
y_te_t = torch.tensor(y_te, dtype=torch.long)

# ── Two-layer neural network ───────────────────────────────────────────────
model = nn.Sequential(
    nn.Linear(64, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10)
)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
loss_fn   = nn.CrossEntropyLoss()
losses    = []

for epoch in range(200):
    model.train()
    pred = model(X_tr_t)
    loss = loss_fn(pred, y_tr_t)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())

# ── Evaluation ──────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    acc = (model(X_te_t).argmax(1) == y_te_t).float().mean().item()

print(f"Test accuracy on real handwritten digits: {acc*100:.1f}%")

# ── Plot training loss ───────────────────────────────────────────────────
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.title("Training Loss — Handwritten Digit Classifier"); plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.tight_layout(); plt.show()

# ── Show a sample prediction ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(digits.images[i], cmap='gray')
    pred_lbl = model(torch.tensor(X_te[:5])).argmax(1)[i].item()
    ax.set_title(f"Pred: {pred_lbl}  True: {y_te[i]}")
    ax.axis('off')
plt.suptitle("Real-World: ATM / Postal Digit Recognition", fontsize=11)
plt.tight_layout(); plt.show()

## Summary

- We built a **baseline classifier** with scikit-learn on synthetic data with numeric and **categorical** features.
- We used **OneHotEncoder** (via `ColumnTransformer`) to encode the categorical feature `color` before training.
- We evaluated the model with **accuracy**, **confusion matrix**, and **classification report** (precision, recall, F1).
- This pipeline (preprocess → model → evaluate) is the same pattern you will use with image data and CNNs in later notebooks.

## 📚 References & Further Reading

**Foundational Papers:**
- LeCun, Bengio & Hinton (2015) — [Deep Learning](https://www.nature.com/articles/nature14539), *Nature*
- Goodfellow et al. (2016) — [Deep Learning Book](https://www.deeplearningbook.org/) (free online)

**PyTorch Docs:**
- [torch.nn — Building Blocks](https://pytorch.org/docs/stable/nn.html)
- [Autograd — Automatic Differentiation](https://pytorch.org/docs/stable/autograd.html)

**State-of-the-Art (2024–2025):**
- Transformers (GPT-4, LLaMA 3) are deep networks with 70B+ parameters
- Deep learning powers real-time translation, medical diagnosis, self-driving cars